# Comparing models on the same data

TorchPP gives every model the same interface, so models can be compared directly. Here we simulate
data from a known two-type Hawkes process, fit four models of increasing flexibility, compare them on
held-out data, and check that the Hawkes models recover the true excitation matrix.

| model | what it can express |
|---|---|
| `poisson` | a constant rate per type, no dependence on the past |
| `const_exp_hawkes` | constant baseline + exponential kernels (the true model class) |
| `spline_spline_hawkes` | spline baseline + spline kernels of any shape |
| `neural_spline` | a transformer encoder of the history + a spline intensity head |

Runs in a few minutes (about 2–4 on a laptop). Background: [`00_tpp_introduction.ipynb`](./00_tpp_introduction.ipynb).

In [ ]:
import torch
import matplotlib.pyplot as plt
import torchpp as tpp
from torchpp.modules.intensity import ConstantBaselineParams, ExpKernelParams
from torchpp.training import default_fit_config

torch.set_default_dtype(torch.float64)

## Simulate data with known ground truth

`TokenRoles(has_origin=True)`: time 0 is a real start, so every event is scored, including the first one.

In [ ]:
mu = torch.tensor([0.3, 0.2])
alpha = torch.tensor([[0.4, 0.0],
                      [0.3, 0.5]])   # [target, source]
beta = torch.tensor([[2.0, 1.0],
                     [1.0, 1.5]])
true_model = tpp.ConstBaselineExpKernelHawkesProcess(
    D=2,
    baseline_params=ConstantBaselineParams(mu=mu),
    kernel_params=ExpKernelParams(alpha=alpha, beta=beta),
    token_roles=tpp.TokenRoles(has_origin=True),
)

T_OBS = 25.0


def simulate(n, seed0):
    seqs = []
    with torch.no_grad():
        for i in range(n):
            s = true_model.sample(None, num_steps=60, rng=seed0 + i).sequence
            keep = s.time_points <= T_OBS
            seqs.append(tpp.MVEventData(s.time_points[keep], s.event_types[keep], T_obs=T_OBS))
    return seqs


train, val, test = simulate(1000, 0), simulate(40, 10_000), simulate(40, 20_000)
print(f"{len(train)} train / {len(val)} val / {len(test)} test sequences, "
      f"{sum(len(s) for s in train) / len(train):.1f} events each on average")

## Fit four models

`DatasetStats.from_sequences` measures the data (time unit, rates); `create_model` uses it for the
time scale and the initial values. `fit` runs mini-batch training with validation and keeps the best
weights.

In [ ]:
stats = tpp.DatasetStats.from_sequences(train, D=2)

setups = {
    "poisson": dict(lr=0.05),
    "const_exp_hawkes": dict(lr=0.05),
    "spline_spline_hawkes": dict(lr=0.05, arch={"kernel": {"num_knots": 8}}),
    "neural_spline": dict(lr=0.01, arch={"encoder": {"n_layers": 2, "n_heads": 2, "n_embd": 16}}),
}
fitted = {}
for name, s in setups.items():
    model = tpp.create_model(name, 2, arch=s.get("arch"), dataset_stats=stats, token_roles=tpp.TokenRoles(has_origin=True), seed=0)
    config = default_fit_config(num_steps=250, learning_rate=s["lr"], eval_freq=50, batch_size=32)
    fitted[name] = tpp.fit(model, train, val, config).model
    print(f"{name:22s} trained")

## Compare on held-out data

`evaluate_tpp` predicts every event of the test sequences from its history and reports, per event:

- `ll_per_event`: the full-sequence log-likelihood per event (the training objective), higher is better.
  Besides the events it scores the empty stretch after the last event up to the end of the window;
- `marginal_time_ll` and `cond_type_ll`: the "when" and "which type" parts of the next-event log-likelihood
  `joint_ll = marginal_time_ll + cond_type_ll` (without that final empty stretch, so they do not add up to
  `ll_per_event`);
- `cond_top1_accuracy`: how often the most likely type at the true time is the right one.

The true model is the best any model can do on average.

In [ ]:
keys = ["ll_per_event", "marginal_time_ll", "cond_type_ll", "cond_top1_accuracy"]
rows = {"true model": tpp.evaluate_tpp(true_model, test, compute_marginals=False)}
for name, model in fitted.items():
    rows[name] = tpp.evaluate_tpp(model, test, compute_marginals=False)

print(f"{'':22s}" + "".join(f"{k:>20s}" for k in keys))
for name, r in rows.items():
    print(f"{name:22s}" + "".join(f"{r[k]:20.3f}" for k in keys))

The Poisson process ignores the history and loses on both the time and the type. The two Hawkes models
come close to the true model. The neural model is flexible enough too, but needs more data and training
to match a model with the right structure.

## Fitted vs true intensity on a held-out sequence

In [ ]:
seq = test[0]
batch = tpp.BatchedMVEventData(mv_events=[seq])
grid = torch.linspace(0, T_OBS, 2000).unsqueeze(0)

fig, axs = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
with torch.no_grad():
    curves = {"true model": true_model.intensity(grid, batch)[0]}
    curves.update({n: m.intensity(grid, batch)[0] for n, m in fitted.items()})
for d in range(2):
    for name, lam in curves.items():
        style = dict(color="k", lw=2) if name == "true model" else dict(lw=1)
        axs[d].plot(grid[0], lam[:, d], label=name, **style)
    ev = seq.time_points[seq.event_types == d]
    axs[d].plot(ev, torch.zeros_like(ev), "|", color="k", markersize=12)
    axs[d].set_ylabel(f"$\\lambda_{d}(t)$")
axs[0].legend(ncol=3, fontsize=8); axs[1].set_xlabel("time")
plt.tight_layout(); plt.show()

## Recover the excitation matrix

`branching_matrix()` returns $\int_0^\infty \phi_{d,j}$: the expected number of type-$d$ events that one
type-$j$ event triggers, indexed `[target, source]`. For the exponential kernel this is $\alpha$. The
spline-kernel model has no $\alpha$ parameter, but its kernels integrate to the same quantity.

In [ ]:
with torch.no_grad():
    print("true alpha:\n", alpha.numpy().round(2))
    for name in ["const_exp_hawkes", "spline_spline_hawkes"]:
        print(f"{name}:\n", fitted[name].branching_matrix().numpy().round(2))

`kernel_values(dt)` gives the kernel shapes themselves, in native time units. The spline kernel is a
piecewise-linear curve with 8 knots here; it was never told that the true kernels are exponential. Each row shares its y-axis, so a
small spurious kernel (the true effect of type 1 on type 0 is zero) also looks small.

In [ ]:
dt = torch.linspace(0.01, 4, 200)
with torch.no_grad():
    true_k = true_model.kernel_values(dt)
    spline_k = fitted["spline_spline_hawkes"].kernel_values(dt)
fig, axs = plt.subplots(2, 2, figsize=(9, 5), sharex=True, sharey="row")
for d in range(2):
    for j in range(2):
        axs[d, j].plot(dt, true_k[:, d, j], "k", label="true")
        axs[d, j].plot(dt, spline_k[:, d, j], label="spline kernel")
        axs[d, j].set_title(f"effect of type {j} on type {d}", fontsize=9)
axs[0, 0].legend(); axs[1, 0].set_xlabel("lag"); axs[1, 1].set_xlabel("lag")
plt.tight_layout(); plt.show()

## Next steps

- Train on the benchmark datasets with `train.py`: [`docs/getting_started.md`](../docs/getting_started.md).
- Fit your own data: the recipe in [`docs/getting_started.md`](../docs/getting_started.md#your-own-data).
- All models: [`docs/models.md`](../docs/models.md).